# PyTorch Tensor Exercises — SOLUTION

*ML & NLP course — Data Trainers LLC — Axel Sirota*

> **Instructor copy.** Every lab is fully implemented with explanatory comments, common pitfalls called out, and alternative approaches noted.

## The story

You're a data scientist who has been working with TensorFlow for years. Your new team runs everything on **PyTorch** — models, pipelines, training loops, the works. This notebook is the solution reference for the eight essential PyTorch tensor topics.

## Learning objectives

1. Create tensors of any shape and dtype with `torch.tensor`, `torch.zeros/ones/rand/randn`.
2. Perform arithmetic, statistical, and matrix operations on tensors.
3. Reshape tensors with `.reshape`, `.view`, `.transpose`, `.permute`, `.flatten`, `.squeeze`/`.unsqueeze`.
4. Index and slice tensors, including boolean masking.
5. Explain broadcasting rules and apply them confidently.
6. Compute gradients with `requires_grad=True` and `loss.backward()`.
7. Build a `TensorDataset` + `DataLoader` pipeline.
8. Assemble a forward pass with `nn.Linear`, `nn.ReLU`, a loss function, and backprop.

## Section 0 — Environment Setup

In [ ]:
# Install required packages (run this first on Google Colab).
!pip install -q torch numpy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ---- Reproducibility ----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---- Device detection ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")

print("\nEnvironment setup complete.")

## Section 1 — Tensor Basics

Factory functions: `torch.tensor`, `torch.zeros`, `torch.ones`, `torch.rand`, `torch.randn`. Key attributes: `.shape`, `.dtype`. NumPy conversion: `torch.from_numpy` (shared memory) vs `torch.tensor(arr)` (copy).

In [ ]:
# Demo: tensor creation
scalar  = torch.tensor(3.14)
vector  = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
matrix  = torch.zeros(3, 3)
rand3d  = torch.randn(2, 3, 4)

for name, t in [('scalar', scalar), ('vector', vector),
                ('matrix', matrix), ('rand3d', rand3d)]:
    print(f"{name:8s}  shape={str(t.shape):20s}  dtype={t.dtype}")

arr  = np.array([[1.0, 2.0], [3.0, 4.0]])
t_np = torch.from_numpy(arr)
back = t_np.numpy()
print(f"\nNumPy -> torch -> NumPy: all values identical: {np.allclose(arr, back)}")

### Solution: Lab 1 — Tensor Basics

In [ ]:
# Solution: Lab 1 — Tensor Basics

np_arr = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

# 1. Scalar tensor with value 7.0
# torch.tensor(value) creates a 0-D tensor; float() ensures float32 dtype
s = torch.tensor(7.0)

# 2. 1-D tensor with float32 dtype
# Pass dtype explicitly — Python ints default to int64 in torch.tensor
v = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0], dtype=torch.float32)

# 3. 2-D tensor of ones, shape (3, 3)
# torch.ones(*shape) — alternatively torch.ones((3, 3)) also works
m = torch.ones(3, 3)

# 4. 3-D random normal tensor
# torch.randn(*shape) draws from N(0,1)
t3d = torch.randn(2, 4, 4)

# 5. NumPy array -> tensor
# torch.from_numpy shares memory with np_arr (zero-copy)
# Alternative: torch.tensor(np_arr) creates an independent copy
t_from_np = torch.from_numpy(np_arr)

# 6. Tensor -> NumPy array
# .numpy() only works on CPU tensors without requires_grad
arr_back = t_from_np.numpy()

# ---- Verification ----
print(f"s          : value={s.item():.1f}  shape={s.shape}  dtype={s.dtype}")
print(f"v          : shape={v.shape}  dtype={v.dtype}  values={v.tolist()}")
print(f"m          : shape={m.shape}  all_ones={m.all().item()}")
print(f"t3d        : shape={t3d.shape}  dtype={t3d.dtype}")
print(f"t_from_np  : shape={t_from_np.shape}  dtype={t_from_np.dtype}")
print(f"arr_back   : type={type(arr_back).__name__}  shape={arr_back.shape}")

# Common pitfall: torch.tensor([10,20,30,40,50]) without dtype gives int64,
# which will break later ops expecting float. Always specify dtype=torch.float32
# when the list contains integers but floats are needed.

## Section 2 — Tensor Operations

Demo + solution for arithmetic, statistics, ReLU, and matrix multiply.

In [ ]:
torch.manual_seed(SEED)

# Demo: operations
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([10.0, 20.0, 30.0, 40.0])

print("a + b  =", (a + b).tolist())
print("a - b  =", (a - b).tolist())
print("a * b  =", (a * b).tolist())
print("a / b  =", (a / b).tolist())
print(f"a.mean() = {a.mean().item():.4f}")
print(f"a.std()  = {a.std().item():.4f}")

c = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print(f"relu({c.tolist()}) = {torch.relu(c).tolist()}")

A = torch.ones(2, 3)
B = torch.ones(3, 2) * 2.0
C = A @ B
print(f"\n(2x3) @ (3x2) = {C.shape}\n{C}")

In [ ]:
# Solution: Lab 2 — Tensor Operations
torch.manual_seed(SEED)

x = torch.tensor([1.0, 4.0, 9.0, 16.0])
y = torch.tensor([2.0, 2.0, 3.0,  4.0])

# 1. Element-wise arithmetic — standard Python operators work directly
add_xy = x + y          # equivalent: torch.add(x, y)
sub_xy = x - y          # equivalent: torch.sub(x, y)
mul_xy = x * y          # equivalent: torch.mul(x, y)
div_xy = x / y          # equivalent: torch.div(x, y)

# 2. Mean and std over a (4,4) random matrix
mat = torch.randn(4, 4)
mat_mean = mat.mean()   # global scalar mean; mat.mean(dim=0) gives per-column
mat_std  = mat.std()    # Bessel-corrected (ddof=1) by default, like NumPy

# 3. ReLU — max(0, x) element-wise
# torch.relu is the functional form; nn.ReLU() is the module form (equivalent here)
relu_out = torch.relu(torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0]))

# 4. Matrix multiply: (3,4) @ (4,5) -> (3,5)
# Each entry of PQ[i,j] = sum over k of P[i,k]*Q[k,j] = 4 * 1 * 3 = 12
P  = torch.ones(3, 4)
Q  = torch.ones(4, 5) * 3.0
PQ = P @ Q   # equivalent: torch.matmul(P, Q) or torch.mm(P, Q) for strict 2-D

print(f"x + y  = {add_xy.tolist()}")
print(f"x - y  = {sub_xy.tolist()}")
print(f"x * y  = {mul_xy.tolist()}")
print(f"x / y  = {div_xy.tolist()}")
print(f"\nmat mean = {mat_mean.item():.4f}  std = {mat_std.item():.4f}")
print(f"relu_out = {relu_out.tolist()}")
print(f"PQ shape = {PQ.shape}  values all 12.0: {(PQ == 12.0).all().item()}")

## Section 3 — Manipulating Shapes

Demo + solution for `reshape`, `view`, `transpose`, `permute`, `flatten`, `squeeze`, `unsqueeze`.

In [ ]:
t = torch.arange(16, dtype=torch.float32)
print("Original:", t.shape)

r1 = t.reshape(4, 4)
print("reshape(4,4)  :", r1.shape)

r3 = r1.transpose(0, 1)
print("transpose(0,1):", r3.shape)

t3 = torch.randn(2, 3, 5)
r4 = t3.permute(2, 0, 1)
print(f"permute(2,0,1) on {t3.shape}: {r4.shape}")

t4 = torch.zeros(2, 1, 3, 1)
print(f"squeeze() {t4.shape} -> {t4.squeeze().shape}")
print(f"unsqueeze(0) {t.shape} -> {t.unsqueeze(0).shape}")
print(f"flatten() {r1.shape} -> {r1.flatten().shape}")

In [ ]:
# Solution: Lab 3 — Manipulating Shapes

base = torch.arange(24, dtype=torch.float32)   # [0..23]
img  = torch.randn(28, 28)

# 1. Reshape to (3, 8) — total elements = 24, 3*8 = 24 ✓
reshaped_3x8 = base.reshape(3, 8)

# 2. Reshape to (2, 3, 4) — 2*3*4 = 24 ✓
reshaped_3d  = base.reshape(2, 3, 4)

# 3. Transpose dims 0 and 1 of reshaped_3x8: (3, 8) -> (8, 3)
# transpose() always returns a VIEW (non-contiguous); call .contiguous() if you need
# to call .view() on the result.
transposed   = reshaped_3x8.transpose(0, 1)

# 4. Permute (2,3,4) to (4,2,3): new order of dims is [2, 0, 1]
permuted     = reshaped_3d.permute(2, 0, 1)

# 5. Flatten to 1-D — equivalent to reshape(-1)
flat         = reshaped_3d.flatten()

# 6. Add batch dim at position 0: (28,28) -> (1,28,28)
# unsqueeze(0) inserts a dim of size 1 at position 0
img_batched  = img.unsqueeze(0)

# 7. Remove that batch dim: (1,28,28) -> (28,28)
# squeeze(0) removes the dim at position 0 if it is size 1
img_back     = img_batched.squeeze(0)

checks = [
    ('reshaped_3x8', reshaped_3x8, (3, 8)),
    ('reshaped_3d',  reshaped_3d,  (2, 3, 4)),
    ('transposed',   transposed,   (8, 3)),
    ('permuted',     permuted,     (4, 2, 3)),
    ('flat',         flat,         (24,)),
    ('img_batched',  img_batched,  (1, 28, 28)),
    ('img_back',     img_back,     (28, 28)),
]
for name, t, expected in checks:
    status = "OK" if tuple(t.shape) == expected else f"WRONG (got {tuple(t.shape)})"
    print(f"{name:15s} shape={tuple(t.shape)}  expected={expected}  [{status}]")

## Section 4 — Indexing and Slicing

Demo + solution. Syntax is identical to NumPy.

In [ ]:
grid = torch.arange(1, 10, dtype=torch.float32).reshape(3, 3)
print("grid:\n", grid)
print("\ngrid[0]       =", grid[0].tolist())
print("grid[0, 2]    =", grid[0, 2].item())
print("grid[1:3, :]  =\n", grid[1:3, :])
print("grid[:, 1]    =", grid[:, 1].tolist())
print("grid[grid>4]  =", grid[grid > 4].tolist())

In [ ]:
# Solution: Lab 4 — Indexing and Slicing
torch.manual_seed(SEED)
data = torch.randn(4, 5) * 5
print("data:\n", data.round(decimals=2))

# 1. Single element: row 1, col 2
# data[1, 2] and data[1][2] are equivalent; the comma form is preferred
elem = data[1, 2]

# 2. Rows 1 and 2 (Python slice notation: start inclusive, stop exclusive)
rows_1_2 = data[1:3, :]   # equivalent: data[1:3]

# 3. Last column — negative index -1 means "last"
last_col = data[:, -1]

# 4. Boolean masking: selects a flat 1-D tensor of matching values
negatives = data[data < 0]

# 5. Compound boolean mask: & requires parens because of Python operator precedence
mid_range = data[(data >= 3) & (data <= 7)]

print(f"\nelem       = {elem.item():.4f}")
print(f"rows_1_2   shape={rows_1_2.shape}")
print(f"last_col   shape={last_col.shape}")
print(f"negatives  = {negatives.round(decimals=2).tolist()}")
print(f"mid_range  = {mid_range.round(decimals=2).tolist()}")

# Common mistake: writing data[data >= 3 and data <= 7] raises a RuntimeError
# because Python's 'and' calls .bool() on the full tensor. Always use & / | operators.

## Section 5 — Broadcasting

Demo + solution. Key rule: dimensions are aligned from the right; size-1 dims are stretched.

In [ ]:
M = torch.ones(3, 4)
row = torch.tensor([0.0, 1.0, 2.0, 3.0])
result1 = M + row
print("(3,4) + (4,) ->", result1.shape)
print(result1)

col  = torch.tensor([[10.0], [20.0], [30.0]])
row2 = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
result2 = col + row2
print("\n(3,1) + (1,4) ->", result2.shape)
print(result2)

try:
    _ = torch.ones(3, 4) + torch.ones(3, 3)
except RuntimeError as e:
    print(f"\nExpected error: {e}")

In [ ]:
# Solution: Lab 5 — Broadcasting
torch.manual_seed(SEED)
features    = torch.randn(6, 5)
bias        = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])   # (5,) -> broadcast to (6,5)
col_scale   = torch.tensor([[2.0], [3.0], [4.0]])          # (3, 1)
row_weights = torch.tensor([[1.0, 0.5, 0.25]])             # (1, 3)

# 1. Add bias to every row
# bias shape (5,) is aligned from the right with features (6, 5).
# It is implicitly treated as (1, 5), then stretched to (6, 5).
biased = features + bias

# 2. Outer-product via broadcasting
# col_scale (3,1) * row_weights (1,3) -> (3,3)
# Entry [i,j] = col_scale[i] * row_weights[j] — a scaling table
scaled = col_scale * row_weights
# Each entry [i,j] means: "scale factor for row i, column j"
# i.e., row 0 gets [2*1, 2*0.5, 2*0.25] = [2.0, 1.0, 0.5]

# 3. Predict and verify broadcast compatibility
ops = [
    ("(4,3)+(4,3)", lambda: torch.ones(4,3) + torch.ones(4,3)),   # same shape -> OK
    ("(4,1)+(1,3)", lambda: torch.ones(4,1) + torch.ones(1,3)),   # both dims broadcast -> (4,3)
    ("(4,3)+(4,2)", lambda: torch.ones(4,3) + torch.ones(4,2)),   # 3 vs 2 -> ERROR
]
for name, op in ops:
    try:
        result = op()
        print(f"{name} -> OK, shape={result.shape}")
    except RuntimeError as e:
        print(f"{name} -> ERROR: {e}")

print(f"\nbiased shape = {biased.shape}  (expected (6, 5))")
print(f"scaled shape = {scaled.shape}  (expected (3, 3))")
print(f"scaled:\n{scaled}")

## Section 6 — Autograd and Differentiation

Demo + solution. PyTorch builds a dynamic computation graph; `.backward()` traverses it via chain rule.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2
loss.backward()
print(f"w={w.item()}, loss={loss.item()}, grad={w.grad.item()}, expected={2*w.item()}")

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
f = x**2 + 3 * x * y
f.backward()
print(f"\nf(2,5)={f.item()}, df/dx={x.grad.item()}, df/dy={y.grad.item()}")
print(f"Expected df/dx={2*x.item()+3*y.item()}, df/dy={3*x.item()}")

with torch.no_grad():
    z = w ** 3
print(f"\nz.requires_grad={z.requires_grad}")

In [ ]:
# Solution: Lab 6 — Autograd

# 1. Gradient of a^3 at a=4
# d/da (a^3) = 3a^2 = 3 * 16 = 48
a      = torch.tensor(4.0, requires_grad=True)
loss_a = a ** 3                # build the computation graph
loss_a.backward()              # compute grad via chain rule
grad_a = a.grad                # retrieve the gradient

# 2. d/dp (p*q + p^2) = q + 2p = -3 + 4 = 1
#    d/dq (p*q + p^2) = p      = 2
p       = torch.tensor(2.0,  requires_grad=True)
q       = torch.tensor(-3.0, requires_grad=True)
loss_pq = p * q + p ** 2       # graph: two leaves p and q
loss_pq.backward()
grad_p  = p.grad
grad_q  = q.grad

# 3. No-grad block — no graph is built, requires_grad of result is False
# Use this during inference to save memory (no need to store activations for backprop)
with torch.no_grad():
    loss_no_grad = p * q + p ** 2  # p and q still have requires_grad, but output won't

print(f"grad_a = {grad_a.item():.1f}  expected = {3*(4.0**2):.1f}  OK={abs(grad_a.item()-48.0)<1e-4}")
print(f"grad_p = {grad_p.item():.1f}  expected = {-3.0+2*2.0:.1f}  OK={abs(grad_p.item()-1.0)<1e-4}")
print(f"grad_q = {grad_q.item():.1f}  expected = {2.0:.1f}  OK={abs(grad_q.item()-2.0)<1e-4}")
print(f"loss_no_grad.requires_grad = {loss_no_grad.requires_grad}  (should be False)")

# Common pitfalls:
# - Calling .backward() twice on the same graph -> RuntimeError (graph freed after first call).
#   Fix: loss.backward(retain_graph=True) if you need to call backward twice.
# - Forgetting zero_grad in a loop -> gradients accumulate across steps.

## Section 7 — Dataset and DataLoader

Demo + solution. `TensorDataset` wraps tensors; `DataLoader` handles batching and shuffling.

In [ ]:
torch.manual_seed(SEED)

N, F = 100, 8
X_demo = torch.randn(N, F)
y_demo = (X_demo[:, 0] > 0).long()

ds_demo = TensorDataset(X_demo, y_demo)
print(f"Dataset length: {len(ds_demo)}")
print(f"First sample: X shape={ds_demo[0][0].shape}, y={ds_demo[0][1].item()}")

loader_demo = DataLoader(ds_demo, batch_size=16, shuffle=True, num_workers=0)
print(f"\nNumber of batches per epoch: {len(loader_demo)}")
x_batch, y_batch = next(iter(loader_demo))
print(f"Batch X shape: {x_batch.shape}   Batch y shape: {y_batch.shape}")

In [ ]:
# Solution: Lab 7 — Dataset and DataLoader
torch.manual_seed(SEED)

# 1. Create feature matrix and labels
# X_lab shape (200, 4); y_lab is a linear function of two features
X_lab = torch.randn(200, 4, dtype=torch.float32)
y_lab = X_lab[:, 0] * 2 + X_lab[:, 1] - 1   # float32 because X_lab is float32

# 2. Wrap in TensorDataset — __getitem__ returns (X_lab[i], y_lab[i]) as a tuple
lab_dataset = TensorDataset(X_lab, y_lab)

# 3. DataLoader: shuffle=True is critical for training (avoids order-dependent bias)
# num_workers=0 is safe on Colab; increase to 2-4 on a local multi-core machine
lab_loader = DataLoader(lab_dataset, batch_size=32, shuffle=True, num_workers=0)

# 4. One-epoch iteration
total_samples = 0
for i, (xb, yb) in enumerate(lab_loader):
    batch_mean_y = yb.mean().item()
    total_samples += xb.shape[0]
    print(f"  Batch {i:2d}: size={xb.shape[0]}, y_mean={batch_mean_y:.4f}")

print(f"\nDataset length : {len(lab_dataset)}  (expected 200)")
print(f"Num batches    : {len(lab_loader)}  "
      f"(200//32 = {200//32} full batches + 1 remainder = {200//32+1})")
print(f"Total samples  : {total_samples}  (expected 200)")

## Section 8 — Simple Neural Network Components

Demo + solution for `nn.Module`, `nn.Linear`, `nn.ReLU`, loss, optimizer, and the full training loop.

In [ ]:
torch.manual_seed(SEED)

demo_net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)
print("Demo model:", demo_net)
total_params = sum(p.numel() for p in demo_net.parameters())
print(f"Total parameters: {total_params}")

x_dummy = torch.randn(5, 4)
logits_demo = demo_net(x_dummy)
print(f"Input  shape: {x_dummy.shape}")
print(f"Logits shape: {logits_demo.shape}")

loss_demo = logits_demo.mean()
loss_demo.backward()
print(f"Gradient on first linear weight: {demo_net[0].weight.grad.shape}")

In [ ]:
# Solution: Lab 8 — Build and Train a Tiny Network
torch.manual_seed(SEED)

# 1. Define SmallNet — a two-hidden-layer regression network
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear(in, out) creates a weight matrix (out, in) and bias (out,)
        self.fc1  = nn.Linear(4, 16)
        self.relu = nn.ReLU()
        # Output is size 1 (regression — predict a single continuous value)
        self.fc2  = nn.Linear(16, 1)

    def forward(self, x):
        # x: (B, 4)
        x = self.fc1(x)    # (B, 4)  -> (B, 16)
        x = self.relu(x)   # (B, 16) -> (B, 16), negatives zeroed
        x = self.fc2(x)    # (B, 16) -> (B, 1)
        return x           # logits / regression output

# 2. Instantiate and move to device
model_lab = SmallNet().to(device)
print(model_lab)
print(f"Parameters: {sum(p.numel() for p in model_lab.parameters())}")

# 3. Loss and optimizer
loss_fn   = nn.MSELoss()                                         # mean squared error for regression
optimizer = torch.optim.Adam(model_lab.parameters(), lr=1e-2)   # Adam: adaptive lr per parameter

# 4 & 5. Training loop — 10 epochs
EPOCHS_LAB = 10
epoch_losses_lab = []

for epoch in range(EPOCHS_LAB):
    model_lab.train()          # enables dropout/batchnorm training mode (safe to call even without them)
    total_loss, n_batches = 0.0, 0

    for xb, yb in lab_loader:
        xb = xb.to(device)
        yb = yb.to(device).unsqueeze(1)   # (B,) -> (B, 1) to match fc2 output shape

        optimizer.zero_grad()             # CRITICAL: clear accumulated gradients from previous step
        preds = model_lab(xb)            # forward pass: (B, 4) -> (B, 1)
        loss  = loss_fn(preds, yb)       # MSE: mean over batch of (pred - true)^2
        loss.backward()                  # backprop: compute d(loss)/d(param) for all params
        optimizer.step()                 # gradient descent step: param -= lr * grad

        total_loss += loss.item()
        n_batches  += 1

    avg = total_loss / max(n_batches, 1)
    epoch_losses_lab.append(avg)
    print(f"Epoch {epoch+1:2d}/{EPOCHS_LAB}  avg MSE loss = {avg:.4f}")

# 6. Inference — use torch.no_grad() to skip building the graph (saves memory)
model_lab.eval()    # switches to inference mode (affects dropout/batchnorm if present)
xb_infer, yb_infer = next(iter(lab_loader))
xb_infer = xb_infer.to(device)
with torch.no_grad():
    preds_infer = model_lab(xb_infer).squeeze(1)  # (B, 1) -> (B,)

print("\nFirst 5 predictions vs ground truth:")
for pred, true in zip(preds_infer[:5].cpu(), yb_infer[:5]):
    print(f"  pred={pred.item():.4f}   true={true.item():.4f}")

# Expected behavior: after 10 epochs with lr=1e-2, loss should drop substantially
# from its initial value (~5-6 MSE on standardized data) to under 1.0.
# The task is linear (y = 2*x0 + x1 - 1), so the network should fit it well.

## Optional Lab Solution — Gradient Descent from Scratch

Raw autograd without any `nn` modules — this is what `optimizer.step()` automates.

In [ ]:
# Solution: Optional Lab — Gradient Descent from Scratch
torch.manual_seed(SEED)

lr    = 0.1
STEPS = 200

# 1. Data: y = 2x + 1 + noise
x_gd = torch.linspace(-1, 1, 50)
y_gd = 2 * x_gd + 1 + 0.1 * torch.randn(50)

# 2. Learnable parameters — plain tensors with requires_grad=True
# This is exactly what nn.Parameter() wraps; we skip the wrapper for clarity.
w_gd = torch.tensor(0.0, requires_grad=True)
b_gd = torch.tensor(0.0, requires_grad=True)

for step in range(STEPS):
    # Forward: linear prediction
    y_pred_gd = w_gd * x_gd + b_gd               # broadcasts x_gd (50,) with scalars

    # Loss: mean squared error
    loss_gd = ((y_pred_gd - y_gd) ** 2).mean()   # scalar

    # Backward: compute d(loss)/d(w) and d(loss)/d(b)
    loss_gd.backward()

    # Gradient descent step — MUST be inside no_grad to avoid building a graph
    # for the update itself (we don't want to differentiate through the update rule)
    with torch.no_grad():
        w_gd -= lr * w_gd.grad    # w = w - lr * dL/dw
        b_gd -= lr * b_gd.grad    # b = b - lr * dL/db

    # Zero gradients AFTER the update — if we zero before, backward gives None
    # Note: .zero_() is in-place; we can't use w_gd.grad = 0 here because
    # PyTorch would replace the grad tensor with a Python int, not a zero tensor
    w_gd.grad.zero_()
    b_gd.grad.zero_()

print(f"Learned w = {w_gd.item():.4f}  (expected ~2.0)")
print(f"Learned b = {b_gd.item():.4f}  (expected ~1.0)")

# What torch.optim.SGD does internally is exactly these three lines:
#   param.data -= lr * param.grad
#   param.grad.zero_()
# Adam adds momentum and adaptive learning rates on top.

## Congratulations!

All eight sections complete. Here is a summary of the key patterns and common pitfalls.

### Key takeaways

| Section | Core skill | Most common mistake |
|---------|-----------|---------------------|
| 1 — Tensor Basics | Factory functions, `.shape`, `.dtype`, numpy ↔ torch | `torch.tensor([1,2,3])` gives `int64`; add `dtype=torch.float32` |
| 2 — Operations | `+/-/*/`, `mean/std`, `relu`, `@` | Forgetting that `@` requires 2-D for `torch.mm`; use `@` or `matmul` for batches |
| 3 — Shapes | `reshape/view/transpose/permute` | `.view()` fails on non-contiguous tensors after `transpose`; use `.contiguous().view()` or just `.reshape()` |
| 4 — Indexing | Slice, comma notation, boolean mask | Using `and` instead of `&` in boolean masks |
| 5 — Broadcasting | Align from right, stretch size-1 | Forgetting that `(4,3) + (4,2)` fails — the trailing dim must match or be 1 |
| 6 — Autograd | `requires_grad`, `.backward()`, `.grad` | Forgetting `zero_grad` → gradients accumulate; calling `backward()` twice → graph freed |
| 7 — DataLoader | `TensorDataset`, `DataLoader`, `shuffle` | Using `num_workers > 0` on Colab → worker hangs |
| 8 — nn.Module | `nn.Linear/ReLU`, 6-line loop, `no_grad` | Applying `softmax` before `CrossEntropyLoss` (double-softmax); forgetting `.to(device)` on data |

### Next steps

- **Notebook 5 — CBOW Word Embeddings**: uses `nn.Embedding`, `Dataset`, and the full training loop.
- **Notebook 8 — MLP Text Classification**: multi-layer network on 130K news headlines.
- PyTorch official tutorials: https://pytorch.org/tutorials/